# BIOSTAT 707 Checkpoint 1

Name: Xiaoning Li
NetID: xl516

The prediction target is in-hospital death. Predictors will use information available during the first 48 hours after ICU admission. Length of stay, survival time, and in-hospital death will not be used as predictors.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from tableone import TableOne


# use paths relative to the repository root
ROOT = Path.cwd()
assert (ROOT / "checkpoint1.ipynb").is_file(), "Notebook must be at the repo root."

DATA = ROOT / "data"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

# check that all set-a records and the outcome file are available
record_files = sorted((DATA / "set-a").glob("*.txt"))
assert len(record_files) == 4000, f"Expected 4000 files, found {len(record_files)}."
assert (DATA / "Outcomes-a.txt").is_file(), "Outcomes-a.txt is missing."

print(f"Record files: {len(record_files)}")

## Data preparation

In [ ]:
def load_set_a(files):
    tables = []

    for file in files:
        df = pd.read_csv(file)
        record_id = int(file.stem)

        # check that the record id matches the file name
        recorded_id = df.loc[df["Parameter"] == "RecordID", "Value"]
        assert len(recorded_id) == 1
        assert recorded_id.iloc[0] == record_id

        # move the record id from a row to a column
        df = df.loc[df["Parameter"] != "RecordID"].copy()
        df.insert(0, "RecordID", record_id)

        # convert elapsed time to minutes
        time = df["Time"].str.split(":", expand=True).astype(int)
        df["time_minutes"] = time[0] * 60 + time[1]
        tables.append(df)

    return pd.concat(tables, ignore_index=True)


long = load_set_a(record_files)
long = long[["RecordID", "Time", "time_minutes", "Parameter", "Value"]]

# check the number of admissions and the observation window
assert long["RecordID"].nunique() == 4000
assert long["time_minutes"].between(0, 48 * 60).all()

long.to_csv(OUT / "set-a_long.csv", index=False)

## Data quality


In [ ]:
# exclude -1 from the summaries and count other negative values separately
check = long.assign(
    missing_code=long["Value"].eq(-1),
    other_negative=long["Value"].lt(0) & long["Value"].ne(-1),
    valid_value=long["Value"].mask(long["Value"].eq(-1))
)

# records counts admissions with an entry, even if the value is missing
quality = check.groupby("Parameter").agg(
    rows=("Value", "size"),
    records=("RecordID", "nunique"),
    missing_code=("missing_code", "sum"),
    missing_na=("Value", lambda x: x.isna().sum()),
    other_negative=("other_negative", "sum"),
    min=("valid_value", "min"),
    median=("valid_value", "median"),
    max=("valid_value", "max")
)

# compare extreme values with the 1st and 99th percentiles
quantiles = (
    check.groupby("Parameter")["valid_value"]
    .quantile([0.01, 0.99])
    .unstack()
)
quantiles.columns = ["p01", "p99"]
quality = quality.join(quantiles)
quality = quality[
    ["rows", "records", "missing_code", "missing_na",
     "other_negative", "min", "p01", "median", "p99", "max"]
]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(quality)

# repeated measurements at the same time can have different values
keys = ["RecordID", "time_minutes", "Parameter"]
print("Exact duplicate rows:", long.duplicated().sum())
print("Repeated record-time-variable rows:", long.duplicated(keys).sum())

### clean and save the long table

In [ ]:
# keep the original values and replace -1 with missing in a separate column
long["Value_clean"] = long["Value"].replace(-1, np.nan)

# use screening limits for this analysis (values not in normal clinical ranges)
bounds = {
    "Height": (50, 250),
    "Temp": (25, 45),
    "pH": (6, 8),
}

invalid = pd.Series(False, index=long.index)

for variable, (lower, upper) in bounds.items():
    invalid |= (
        long["Parameter"].eq(variable)
        & long["Value_clean"].notna()
        & ~long["Value_clean"].between(lower, upper)
    )

# treat zero weight as missing
invalid |= (
    long["Parameter"].eq("Weight")
    & long["Value_clean"].eq(0)
)

long.loc[invalid, "Value_clean"] = np.nan

# count original missing codes and newly excluded values separately
cleaning_summary = (
    long.assign(
        missing_code=long["Value"].eq(-1),
        excluded=invalid
    )
    .groupby("Parameter")[["missing_code", "excluded"]]
    .sum()
)

display(cleaning_summary.loc[cleaning_summary.sum(axis=1) > 0])

long.to_csv(OUT / "set-a_long.csv", index=False)

I replaced -1 with missing values. I also treated height outside 50–250 cm, temperature outside 25–45°C, pH outside 6–8, and zero weight as missing. I kept the original values so the changes can be checked.

I checked some unusual temperature records against nearby measurements. I treated those values as likely recording errors. I kept other extreme values and repeated timestamps rather than automatically deleting them.

## Wide table

I used one summary for the full 48 hours, with one row per admission.

For each continuous time-series variable, I calculated count, first, last, minimum, maximum, and mean using valid values. I did not impute missing values. Count is zero when there are no valid observations, while the other summaries remain missing. For MechVent, no record is not treated as confirmed absence of ventilation.

In [ ]:
record_ids = pd.Index(
    [int(file.stem) for file in record_files],
    name="RecordID"
)

# keep one value per admission for the static variables
static_vars = ["Age", "Gender", "Height", "ICUType"]

static_rows = long.loc[long["Parameter"].isin(static_vars)]
assert not static_rows.duplicated(["RecordID", "Parameter"]).any()

static = static_rows.pivot(
    index="RecordID",
    columns="Parameter",
    values="Value_clean"
).reindex(record_ids)

# sort by time and keep the original order when timestamps match
series = long.loc[
    ~long["Parameter"].isin(static_vars + ["MechVent"])
].sort_values(["RecordID", "time_minutes"], kind="stable")

# summarize valid measurements over 48 hours, with first and last skipping missing values
summary = (
    series.groupby(["RecordID", "Parameter"])["Value_clean"]
    .agg(["count", "first", "last", "min", "max", "mean"])
    .unstack("Parameter")
)

summary.columns = [
    f"{variable}_{stat}" for stat, variable in summary.columns
]

wide = static.join(summary).reindex(record_ids)

# use zero for no valid measurements and leave the other summaries missing
count_cols = wide.columns[wide.columns.str.endswith("_count")]
wide[count_cols] = wide[count_cols].fillna(0).astype(int)

# summarize ventilation separately without treating no record as no ventilation
vent = (
    long.loc[long["Parameter"].eq("MechVent")]
    .groupby("RecordID")["Value_clean"]
    .agg(["count", "max"])
    .rename(columns={
        "count": "MechVent_count",
        "max": "MechVent_ever_recorded"
    })
)

wide = wide.join(vent)
wide["MechVent_count"] = wide["MechVent_count"].fillna(0).astype(int)

# check that the outcome file has exactly one row for each admission
outcomes = pd.read_csv(DATA / "Outcomes-a.txt")
assert outcomes["RecordID"].is_unique
assert set(outcomes["RecordID"]) == set(record_ids)
assert outcomes["In-hospital_death"].isin([0, 1]).all()

wide = wide.reset_index().merge(
    outcomes,
    on="RecordID",
    how="left",
    validate="one_to_one"
)
wide.columns.name = None

assert len(wide) == 4000
assert wide["RecordID"].is_unique

wide.to_csv(OUT / "set-a_wide.csv", index=False)

## Table 1

In [ ]:
from tableone import TableOne

table_data = wide.copy()

# label the categories and show missing gender as unknown
table_data["Gender"] = table_data["Gender"].map({
    0: "Female",
    1: "Male"
}).fillna("Unknown")

table_data["ICUType"] = table_data["ICUType"].map({
    1: "Coronary care",
    2: "Cardiac surgery recovery",
    3: "Medical",
    4: "Surgical"
})
table_data["Outcome"] = table_data["In-hospital_death"].map({
    0: "Survived",
    1: "Died"
})

columns = [
    "Age", "Gender", "ICUType", "Height", "Weight_first",
    "HR_mean", "Temp_mean", "MAP_mean", "GCS_first",
    "Creatinine_mean", "SAPS-I", "SOFA"
]
categorical = ["Gender", "ICUType"]
continuous = [col for col in columns if col not in categorical]

# exclude missing score codes from the summaries
table_data[["SAPS-I", "SOFA"]] = (
    table_data[["SAPS-I", "SOFA"]].replace(-1, np.nan)
)

# report medians and quartiles for continuous variables, grouped by outcome
table1 = TableOne(
    table_data,
    columns=columns,
    categorical=categorical,
    nonnormal=continuous,
    groupby="Outcome",
    overall=True,
    missing=True,
    pval=False,
    rename={
        "Weight_first": "Weight, first valid (kg)",
        "HR_mean": "Heart rate, 48-h mean",
        "Temp_mean": "Temperature, 48-h mean",
        "MAP_mean": "Invasive MAP, 48-h mean",
        "GCS_first": "GCS, first valid",
        "Creatinine_mean": "Creatinine, 48-h mean"
    }
)

display(table1)

Table 1 compares admissions by in-hospital death. Continuous variables are shown as medians and quartiles, and categorical variables as counts and percentages. Unknown gender is shown as a separate category.

The group that died had a higher median age, higher severity scores, and higher creatinine summaries. Height and invasive MAP had substantial missingness.

## Outcomes

In [ ]:
death = wide["In-hospital_death"]

outcome_summary = pd.DataFrame({
    "n": [
        death.eq(0).sum(),
        death.eq(1).sum(),
        death.isna().sum()
    ]
}, index=["Survived", "Died", "Missing"])

# use all admissions as the denominator
outcome_summary["Percent"] = (
    outcome_summary["n"] / len(wide) * 100
).round(2)

display(outcome_summary)

# exclude -1 from the length of stay summary
los = wide["Length_of_stay"].replace(-1, np.nan)

# length of stay runs from ICU admission to the end of hospitalization
los_summary = pd.DataFrame({
    "n": [los.count()],
    "Missing": [los.isna().sum()],
    "Median": [los.median()],
    "Q1": [los.quantile(0.25)],
    "Q3": [los.quantile(0.75)],
    "Min": [los.min()],
    "Max": [los.max()]
}, index=["Length of stay (days)"])

display(los_summary)

Most admissions ended in survival, and no in-hospital death labels were missing. Length of stay had missing values. I excluded its -1 codes from the summary.

## Missingness

In [ ]:
# leave static variables out of the time-based missingness map
static_vars = ["Age", "Gender", "Height", "ICUType"]
time_vars = sorted(set(long["Parameter"]) - set(static_vars))

observed = long.loc[
    long["Parameter"].isin(time_vars)
    & long["Value_clean"].notna()
].copy()

# use 6-hour bins and include exactly 48 hours in the last bin
observed["time_bin"] = (
    observed["time_minutes"] // 360
).clip(upper=7).astype(int)

# count each admission once per variable and time bin
measured = (
    observed.groupby(["Parameter", "time_bin"])["RecordID"]
    .nunique()
    .unstack("time_bin")
    .reindex(index=time_vars, columns=range(8))
    .fillna(0)
)

# missing means no valid measurement within that time bin
temporal_missing = 100 * (1 - measured / len(wide))
temporal_missing.columns = [
    "0–6", "6–12", "12–18", "18–24",
    "24–30", "30–36", "36–42", "42–48"
]

fig, ax = plt.subplots(figsize=(10, 12))
sns.heatmap(
    temporal_missing,
    cmap="Blues",
    vmin=0,
    vmax=100,
    cbar_kws={"label": "Admissions without a valid measurement (%)"},
    ax=ax
)
ax.set_xlabel("Hours since ICU admission")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(OUT / "missingness_time.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# use the mean columns to avoid repeating the same missingness pattern
mean_cols = sorted(
    col for col in wide.columns if col.endswith("_mean")
)
feature_cols = static_vars + mean_cols + ["MechVent_ever_recorded"]

missing = wide.set_index("RecordID")[feature_cols].isna()

# sort admissions from fewest to most missing variables
missing = missing.loc[
    missing.sum(axis=1).sort_values(kind="stable").index
]

labels = [
    col.removesuffix("_mean")
    .replace("MechVent_ever_recorded", "MechVent")
    for col in feature_cols
]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    missing.astype(int),
    cmap=["#eeeeee", "#2166ac"],
    vmin=0,
    vmax=1,
    yticklabels=False,
    xticklabels=labels,
    cbar_kws={"ticks": [0, 1]},
    ax=ax
)

colorbar = ax.collections[0].colorbar
colorbar.set_ticklabels(["Observed", "Missing"])
ax.set_xlabel("")
ax.set_ylabel("Admissions sorted by number of missing variables")
plt.xticks(rotation=90)
fig.tight_layout()
fig.savefig(OUT / "missingness_wide.png", dpi=150, bbox_inches="tight")
plt.show()

The time map shows the percentage of admissions without a valid measurement in each six-hour interval. Heart rate, GCS, and temperature were recorded more consistently than cholesterol and troponins.

The wide-table map shows missingness across the full 48 hours, using one summary column per variable. ALP, ALT, and AST show similar missingness patterns.

## Missingness and mortality

In [ ]:
variables = sorted(long["Parameter"].unique())
record_ids = pd.Index(wide["RecordID"], name="RecordID")

# mark whether each admission has at least one valid value for each variable
available = (
    long.loc[long["Value_clean"].notna()]
    .groupby(["RecordID", "Parameter"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=record_ids, columns=variables, fill_value=0)
    .gt(0)
)

death = wide.set_index("RecordID")["In-hospital_death"]
results = []

# compare mortality between admissions with and without valid measurements
for variable in variables:
    measured = available[variable]
    n_measured = int(measured.sum())
    n_missing = int((~measured).sum())

    # mortality stays missing if a group has no admissions
    results.append({
        "Variable": variable,
        "Measured_n": n_measured,
        "Unmeasured_n": n_missing,
        "Measured_deaths": int(death.loc[measured].sum()),
        "Unmeasured_deaths": int(death.loc[~measured].sum()),
        "Measured_mortality_pct": (
            100 * death.loc[measured].mean()
        ),
        "Unmeasured_mortality_pct": (
            100 * death.loc[~measured].mean()
        )
    })

mortality_by_missingness = pd.DataFrame(results).set_index("Variable")

# calculate unmeasured minus measured mortality in percentage points
mortality_by_missingness["Difference_pp"] = (
    mortality_by_missingness["Unmeasured_mortality_pct"]
    - mortality_by_missingness["Measured_mortality_pct"]
)

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(mortality_by_missingness.round(2))

mortality_by_missingness.to_csv(OUT / "mortality_by_missingness.csv")

Mortality differed between admissions with and without valid measurements. For example, mortality was higher when troponin I or lactate was recorded, while the pattern was reversed for respiratory rate.

This suggests that missingness may carry information about the outcome. They do not show that testing causes a change in mortality or establish a specific missing-data mechanism.

## AI use

I used ChatGPT to help understand the assignment requirements, set up the environment, revise Python code and draft explanations. It helped me understand github repo set-up process, pixi set-up, long and wide tables, missingness, and outcome comparisons.

The tool made coding and troubleshooting faster but its code still needed checking. It can be unnecessarily complicated, and its code comments can be unnecessarily long.

I reviewed the code section by section, checked the tables and plots, compared selected unusual temperature values with nearby records, and restarted the kernel to run all cells.

## References

Silva I, Moody G, Scott DJ, Celi LA, Mark RG (2012). Predicting in-hospital mortality of ICU patients: The PhysioNet/Computing in Cardiology Challenge 2012. Computing in Cardiology, 39, 245–248.

[PhysioNet Challenge 2012 dataset](https://physionet.org/content/challenge-2012/1.0.0/)